# Soluciones de los ejercicios

Soluciones del cuaderno `01_Del_dato_solar_al_caption.ipynb`. Intenta los ejercicios antes de mirar esto. Este cuaderno es independiente: carga sus propios datos desde `../03_Datos`.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np

DATOS = Path("../03_Datos")
recorte = np.load(DATOS / "recorte_AR11944_20140107_1824.npz")
canales = [str(c) for c in recorte["channels"]]
crudo = recorte["raw"]
constantes = json.loads((DATOS / "normalizacion_constantes.json").read_text())["constants"]
tokens = np.load(DATOS / "tokens_recorte_AR11944_20140107_1824.npy").astype(np.float32)

## Ejercicio 1: normalizar `hmi_m`

In [ ]:
k = constantes["hmi_m"]
x_m = crudo[canales.index("hmi_m")].astype(np.float64)
paso1_m = x_m * k["sl_scale_factor"]
paso2_m = np.sign(paso1_m) * np.log1p(np.abs(paso1_m))
paso3_m = (paso2_m - k["mean"]) / (k["std"] + k["epsilon"])

print(f"mín {paso3_m.min():.2f}   máx {paso3_m.max():.2f}")
assert paso3_m.min() < 0 < paso3_m.max()

El logaritmo con signo conserva la polaridad porque trata la magnitud y el signo por separado: `sign(x)` guarda el signo y `log1p(|x|)` comprime solo la magnitud. Después, la media global de `hmi_m` es casi cero (`mean ≈ 0`), así que al restarla los valores negativos siguen siendo negativos.

## Ejercicio 2: pooling con `tam_salida = 8`

In [ ]:
def pooling_adaptativo(x, tam_salida=16):
    alto, ancho, dim = x.shape
    salida = np.empty((tam_salida, tam_salida, dim), dtype=np.float64)
    for i in range(tam_salida):
        f0, f1 = (i * alto) // tam_salida, -(-((i + 1) * alto) // tam_salida)
        for j in range(tam_salida):
            c0, c1 = (j * ancho) // tam_salida, -(-((j + 1) * ancho) // tam_salida)
            salida[i, j] = x[f0:f1, c0:c1].mean(axis=(0, 1))
    return salida


agrupados_8 = pooling_adaptativo(tokens, 8)
print("forma:", agrupados_8.shape, "-> tokens que ve el modelo de lenguaje:", 8 * 8)
print(f"cada token promedia unos {45 / 8:.1f} x {45 / 8:.1f} tokens de Surya, es decir, unos {45 / 8 * 16:.0f} x {45 / 8 * 16:.0f} píxeles")
assert agrupados_8.shape == (8, 8, 1280)

Con 8 × 8 el modelo ve 64 tokens en lugar de 256: la secuencia es 4 veces más corta (y más barata de procesar), pero cada token mezcla un área mayor y se pierde resolución espacial. Es un compromiso: con 16 × 16 conservamos más detalle de la estructura de la región.

## Ejercicio 3: cuántos tokens ocupa una región

In [ ]:
def lado_en_tokens(inicio_px, fin_px, parche=16):
    """Número de tokens que cubre el tramo [inicio_px, fin_px) de la imagen."""
    return -(-fin_px // parche) - inicio_px // parche


tokens_512 = lado_en_tokens(0, 512)       # caja alineada con la cuadrícula de 16 px
tokens_704_alineada = lado_en_tokens(0, 704)
tokens_704_ar11944 = lado_en_tokens(1672, 2376)   # la caja real de la región, que no empieza en un múltiplo de 16

print("512 px alineada      :", tokens_512, "tokens por lado ->", tokens_512 ** 2, "tokens en total")
print("704 px alineada      :", tokens_704_alineada, "tokens por lado ->", tokens_704_alineada ** 2)
print("704 px de la AR 11944:", tokens_704_ar11944, "tokens por lado ->", tokens_704_ar11944 ** 2)
assert tokens_512 == 32

Una caja de 512 px ocupa 32 × 32 = 1024 tokens si empieza en un múltiplo de 16. La de 704 px ocuparía 44 × 44 = 1936 si estuviera alineada, pero la caja real de la AR 11944 empieza en x = 1672 (no es múltiplo de 16) y por eso toca **45** tokens por lado: se redondea hacia abajo el inicio y hacia arriba el final.

## Ejercicio 4: ¿aprendió algo sobre el hemisferio?

In [ ]:
resultados = json.loads((DATOS / "resultados_heldout.json").read_text())


def hemisferio(texto):
    m = re.search(r"(northern|southern) hemisphere", texto)
    return m.group(1) if m else None


for hemi in ["northern", "southern"]:
    sub = [r for r in resultados if hemisferio(r["reference"]) == hemi]
    aciertos = sum(hemisferio(r["generated"]) == hemi for r in sub)
    print(f"regiones del hemisferio {hemi:9s}: n = {len(sub):2d} | acierta el modelo: {aciertos:2d} | 'siempre norte' acierta: {len(sub) if hemi == 'northern' else 0:2d}")

dice_sur = sum(hemisferio(r["generated"]) == "southern" for r in resultados)
hay_sur = sum(hemisferio(r["reference"]) == "southern" for r in resultados)
print(f"
El modelo escribe 'southern' en {dice_sur} de {len(resultados)} captions; solo {hay_sur} regiones son del sur.")

Un porcentaje de aciertos total engaña cuando las clases están desbalanceadas: si casi todas las regiones están en el norte, "siempre norte" sale muy bien sin mirar la imagen. Dos pruebas mejores:

1. **Desglosar por clase** (lo que hace la celda de arriba): un modelo que de verdad lee la imagen debería acertar en *ambos* hemisferios; "siempre norte" acierta en uno y falla el otro por completo.
Lo que dice la celda: el modelo acierta **todas** las regiones del sur, pero solo 26 de 43 del norte, y escribe "southern" en 29 de 55 captions aunque solo 12 regiones lo son. Es decir, el modelo **tiende a decir "sur"**: eso explica por qué pierde contra "siempre norte" en conjuntos con muchas regiones del norte, y también por qué un porcentaje total no basta. No sabemos por qué (el entrenamiento tuvo pocas regiones del sur o el modelo se agarra a otra pista); la celda solo lo mide.

2. **Cambiar la imagen** y repetir: si la precisión del hemisferio no baja al darle la imagen de otra región, el modelo no está usando la imagen. Es el control que usamos con la pérdida (la "diferencia de imagen").